# ASW Model

### Import Packages:

In [ ]:
import blpapi
import matplotlib.pyplot as plt
import pdblp
import datetime as dt
import pandas as pd
import numpy as np
import patsy
import statsmodels.tsa.stattools as ts
import statsmodels.api as sm
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn import linear_model
import itertools
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.stattools import grangercausalitytests
from statsmodels.regression.rolling import RollingOLS

from datetime import timedelta
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant


Aux function

In [ ]:
def bbg_download_clean_df(ticker_list, fields, start_date, end_date, period = None):
    start_date, end_date = start_date.strftime('%Y%m%d'), end_date.strftime('%Y%m%d')
    if period == None:
        tmp = con.bdh(ticker_list, fields, start_date, end_date)
    else:
        tmp = con.bdh(ticker_list, fields, start_date, end_date, elms = [('periodicitySelection', period)])
        
    tmp = tmp[ticker_list]
    #tmp.columns = tmp.columns.droplevel(1)
    tmp = tmp.sort_values(by ='date', ascending = False)
    return tmp


def check_df_order(df, sort_order = 'descending'):
    if (sort_order =='descending') and (df.index[0] > df.index[1]):
        df = df.sort_values(by = 'date', ascending = False)
    else:
        df = df.sort_values(by = 'date', ascending = True)
    return df

def bdp(ticker_list, fld):
    fld = [fld]
    bdp_df = con.ref(ticker_list, fld)
    bdp_df = bdp_df.set_index('ticker').drop(columns = ['field'])
    bdp_df.columns = fld
    bdp_df.index = ticker_list
    return bdp_df


def correlation(df, diff = False):
    fig, ax = plt.subplots(figsize=(15,15))  
    if diff == True:
        df = check_df_order(df, 'descending')
        corr = df.diff().corr()
        return sns.heatmap(corr, annot=True, linewidths=.5, ax = ax)
    else:
        corr = df.corr()
        return sns.heatmap(corr, annot=True, linewidths=.5, ax = ax)
    
def rolling_correl(df, rolling_period,  x, y, plot = True, differences = False):
    tit = y + ' ' + str(rolling_period)+"dd corr vs" + ''+ x 
    df = check_df_order(df, 'ascending')
    df = df.fillna(method = 'ffill')
    if differences == True:
        df = df.diff()
        tit = y + ' ' + str(rolling_period)+"dd corr vs" + ''+ x +  " diff"
        
    corr = df[y].rolling(rolling_period).corr(df[x])
    if plot == True:
        corr.plot(figsize = (12, 6), title = tit)
    return corr 
    
def scatter(df, y_, x_, equation = True, differences = False):
    if differences == True:
        df = check_df_order(df, 'ascending')
        df = df.fillna(method = 'ffill')
        df = df.diff()
        df = df.dropna()
        
    reg = linear_model.LinearRegression()
    X = df[x_].values.reshape(-1,1)
    y = df[y_].values.reshape(-1,1)
    reg.fit(X, y)
    alpha, beta = reg.intercept_, reg.coef_[0]
    r_sq = reg.score(X,y)
    scatter = sns.lmplot(x_, y_, df, fit_reg=True)
    props = dict(boxstyle='round', alpha=0.5,color=sns.color_palette()[0])
    textstr = "R2= " + str(round(r_sq, 2))
    scatter.ax.text(0, 0, textstr, transform= scatter.ax.transAxes, fontsize= 10, bbox=props)
    return scatter


def stationarity_test(df):
    df = df.dropna()
    res = pd.DataFrame()
    for c in df.columns:
        tmp = ts.adfuller(df[c])[1]
        tmp = pd.DataFrame(tmp, index = [c], columns = ['P_val'])
        res = pd.concat([res, tmp])
    res['below_5%'] = res.apply(lambda x: x < .05)
    return res   
    
    
def create_combos(list_):
    dic_res = {}
    for i in range(1, len(list_)+ 1):
        dic_res[i] = list(itertools.combinations(list_, i))
    return dic_res
    

def grab_filtered_df(list_, df):
    dic_res = {}
    combo_dic = create_combos(list_)
    for k in combo_dic.keys():
        list_res = []
        for i in range(len(combo_dic[k])): 
            tmp = df[list(combo_dic[k][i])]
            list_res.append(tmp)
        dic_res[k] = list_res
    return dic_res


def OLS_stat_m(x, y, intercept = False, summary= False):
    if intercept == True:
        x = sm.add_constant(x)
    model = sm.OLS(y, x)
    fit = model.fit()
    params = fit.params
    r_2 = fit.rsquared
    r_2adj = fit.rsquared_adj
    p_vals = fit.pvalues
    std_errors = fit.scale**.5
    if summary == True:
        print(fit.summary())
    if intercept == True:
        coeff = params
        return coeff, p_vals, round(r_2, 5), round(r_2adj, 5), std_errors
    else:
        beta = params[0]
        return  beta, p_vals, r_2, r_2adj, std_errors
    
    
def generate_models_combos(df, x_to_test, y_var):
    df_to_test = grab_filtered_df(x_to_test, df)
    final = pd.DataFrame()
    cols = ['coeff', 'p_vals','r_sq', 'r_sq_Adj', 'std_err']
    for k in  df_to_test.keys():
        res = pd.DataFrame()
        for i in range(len(df_to_test[k])):
            tmp = OLS_stat_m(df_to_test[k][i], y_var, intercept = True)
            tmp_df = pd.DataFrame([tmp], columns = cols, index = ['- '.join(df_to_test[k][i].columns)])
            res = pd.concat([res, tmp_df])
        final = pd.concat([final, res])
        
    final = final.sort_values(by = 'r_sq_Adj', ascending = False)
    return final


def grangers_causation_matrix(data, variables, test='ssr_chi2test', verbose=False, lagmax = 4):    
    """Check Granger Causality of all possible combinations of the Time series.
    The rows are the response variable, columns are predictors. The values in the table 
    are the P-Values. P-Values lesser than the significance level (0.05), implies 
    the Null Hypothesis that the coefficients of the corresponding past values is 
    zero, that is, the X does not cause Y can be rejected.

    data      : pandas dataframe containing the time series variables
    variables : list containing names of the time series variables.
    """
    df = pd.DataFrame(np.zeros((len(variables), len(variables))), columns=variables, index=variables)
    for c in df.columns:
        for r in df.index:
            test_result = grangercausalitytests(data[[r, c]], maxlag= lagmax, verbose=False)
            p_values = [round(test_result[i+1][0][test][1],4) for i in range(lagmax)]
            min_p_value = np.min(p_values)
            df.loc[r, c] = min_p_value
    df.columns = [var + '_x' for var in variables]
    df.index = [var + '_y' for var in variables]
    return df


def fit_model(df, x_vars_list, y_var_list, constant = True, plot_res =False, summary = True, robust_std_err = False):
    X = df[x_vars_list]
    y = df[y_var_list]
    if constant == True:
        X = sm.add_constant(X)
    model = sm.OLS(y,X)
    fit = model.fit()
    if robust_std_err == True:
        fit=  model.fit(cov_type='HAC',cov_kwds={'maxlags':1})
    if summary == True:
        print(fit.summary())
    res = pd.DataFrame(fit.fittedvalues, columns = ['y_est'])
    res = pd.concat([res, y], axis = 1)
    res['residuals'] = res[y_var_list[0]] - res['y_est']
    if plot_res == True:
        res[[y_var_list[0], 'y_est']].plot(figsize = (16, 5), title ='fitted vs observed')
        plot_acf(res['residuals'], lags = 50)
    
    return {
        'results': res, 
        'fit': fit, 
        'paramns': fit.params,
        'R2': fit.rsquared, 
        'SE': fit.bse
    }



def rolling_reg(df, y_var, x_vars, window, intercept = True, plot =True):
    df_ = check_df_order(df, 'ascending')
    exog_fact = df_[x_vars]
    if intercept == True:
        exog_vars = sm.add_constant(exog_fact)
    
    roll_reg = RollingOLS(df_[y_var], exog_vars, window = window)
    res = roll_reg.fit()
    res_params = res.params
    
    df_['const'] = 1
    y_hat_df = res.params * df_
    y_hat_df['y_hat'] = y_hat_df.sum(axis = 1)
    
    final_df = pd.concat([res_params, y_hat_df['y_hat']], axis = 1)
    final_df[y_var] = df_[y_var]
    final_df['res'] = final_df[y_var[0]] - final_df['y_hat']
    final_df = check_df_order(final_df, 'descending')
    final_df['SSR_total'] = (final_df['res']**2).cumsum()   
    final_df['SSR'] = (final_df['res']**2).rolling(window).sum()
    final_df['r_2'] = res.rsquared
    
    final_df = final_df.dropna() 
    
    if plot == True:
        final_df[[y_var[0], 'y_hat']].plot(figsize = [14, 6], title ='fitted vs obs')
    return final_df.dropna()

def optimal_rolling_window(df, y_var, x_vars, roll_window_list, plot = True):
    res_list = []
    for i in roll_window_list:
        tmp = rolling_reg(df, y_var, x_vars, i,  plot = False)
        res_list.append(tmp['SSR'][0])
    res_df = pd.DataFrame(res_list, index = roll_window_list, columns = ['SSR'])
    res_df.index.name ='roll_per'
    best_ = res_df[res_df.values == res_df.min()[0]].index[0]
    print('Optimal Roll. period equal to ' + str(best_))
    res_df.plot(figsize =(14, 6), title = 'SSR vs Rolling Windows')
    return res_df


In [ ]:
def bbg_download_clean_df(ticker_list, fields, start_date, end_date, period = None):
    start_date, end_date = start_date.strftime('%Y%m%d'), end_date.strftime('%Y%m%d')
    if period == None:
        tmp = con.bdh(ticker_list, fields, start_date, end_date)
    else:
        tmp = con.bdh(ticker_list, fields, start_date, end_date, elms = [('periodicitySelection', period)])
        
    tmp = tmp[ticker_list]
    #tmp.columns = tmp.columns.droplevel(1)
    tmp = tmp.sort_values(by ='date', ascending = False)
    return tmp


def check_df_order(df, sort_order = 'descending'):
    if (sort_order =='descending') and (df.index[0] > df.index[1]):
        df = df.sort_values(by = 'date', ascending = False)
    else:
        df = df.sort_values(by = 'date', ascending = True)
    return df

def bdp(ticker_list, fld):
    fld = [fld]
    bdp_df = con.ref(ticker_list, fld)
    bdp_df = bdp_df.set_index('ticker').drop(columns = ['field'])
    bdp_df.columns = fld
    bdp_df.index = ticker_list
    return bdp_df


def correlation(df, diff = False):
    fig, ax = plt.subplots(figsize=(15,15))  
    if diff == True:
        df = check_df_order(df, 'descending')
        corr = df.diff().corr()
        return sns.heatmap(corr, annot=True, linewidths=.5, ax = ax)
    else:
        corr = df.corr()
        return sns.heatmap(corr, annot=True, linewidths=.5, ax = ax)
    
def rolling_correl(df, rolling_period,  x, y, plot = True, differences = False):
    tit = y + ' ' + str(rolling_period)+"dd corr vs" + ''+ x 
    df = check_df_order(df, 'ascending')
    df = df.fillna(method = 'ffill')
    if differences == True:
        df = df.diff()
        tit = y + ' ' + str(rolling_period)+"dd corr vs" + ''+ x +  " diff"
        
    corr = df[y].rolling(rolling_period).corr(df[x])
    if plot == True:
        corr.plot(figsize = (12, 6), title = tit)
    return corr 
    
def scatter(df, y_, x_, equation = True, differences = False):
    if differences == True:
        df = check_df_order(df, 'ascending')
        df = df.fillna(method = 'ffill')
        df = df.diff()
        df = df.dropna()
        
    reg = linear_model.LinearRegression()
    X = df[x_].values.reshape(-1,1)
    y = df[y_].values.reshape(-1,1)
    reg.fit(X, y)
    alpha, beta = reg.intercept_, reg.coef_[0]
    r_sq = reg.score(X,y)
    scatter = sns.lmplot(x_, y_, df, fit_reg=True)
    props = dict(boxstyle='round', alpha=0.5,color=sns.color_palette()[0])
    textstr = "R2= " + str(round(r_sq, 2))
    scatter.ax.text(0, 0, textstr, transform= scatter.ax.transAxes, fontsize= 10, bbox=props)
    return scatter


def stationarity_test(df):
    df = df.dropna()
    res = pd.DataFrame()
    for c in df.columns:
        tmp = ts.adfuller(df[c])[1]
        tmp = pd.DataFrame(tmp, index = [c], columns = ['P_val'])
        res = pd.concat([res, tmp])
    res['below_5%'] = res.apply(lambda x: x < .05)
    return res   
    
    
def create_combos(list_):
    dic_res = {}
    for i in range(1, len(list_)+ 1):
        dic_res[i] = list(itertools.combinations(list_, i))
    return dic_res
    

def grab_filtered_df(list_, df):
    dic_res = {}
    combo_dic = create_combos(list_)
    for k in combo_dic.keys():
        list_res = []
        for i in range(len(combo_dic[k])): 
            tmp = df[list(combo_dic[k][i])]
            list_res.append(tmp)
        dic_res[k] = list_res
    return dic_res


def OLS_stat_m(x, y, intercept = False, summary= False):
    if intercept == True:
        x = sm.add_constant(x)
    model = sm.OLS(y, x)
    fit = model.fit()
    params = fit.params
    r_2 = fit.rsquared
    r_2adj = fit.rsquared_adj
    p_vals = fit.pvalues
    std_errors = fit.scale**.5
    if summary == True:
        print(fit.summary())
    if intercept == True:
        coeff = params
        return coeff, p_vals, round(r_2, 5), round(r_2adj, 5), std_errors
    else:
        beta = params[0]
        return  beta, p_vals, r_2, r_2adj, std_errors
    
    
def generate_models_combos(df, x_to_test, y_var):
    df_to_test = grab_filtered_df(x_to_test, df)
    final = pd.DataFrame()
    cols = ['coeff', 'p_vals','r_sq', 'r_sq_Adj', 'std_err']
    for k in  df_to_test.keys():
        res = pd.DataFrame()
        for i in range(len(df_to_test[k])):
            tmp = OLS_stat_m(df_to_test[k][i], y_var, intercept = True)
            tmp_df = pd.DataFrame([tmp], columns = cols, index = ['- '.join(df_to_test[k][i].columns)])
            res = pd.concat([res, tmp_df])
        final = pd.concat([final, res])
        
    final = final.sort_values(by = 'r_sq_Adj', ascending = False)
    return final


def grangers_causation_matrix(data, variables, test='ssr_chi2test', verbose=False, lagmax = 4):    
    """Check Granger Causality of all possible combinations of the Time series.
    The rows are the response variable, columns are predictors. The values in the table 
    are the P-Values. P-Values lesser than the significance level (0.05), implies 
    the Null Hypothesis that the coefficients of the corresponding past values is 
    zero, that is, the X does not cause Y can be rejected.

    data      : pandas dataframe containing the time series variables
    variables : list containing names of the time series variables.
    """
    df = pd.DataFrame(np.zeros((len(variables), len(variables))), columns=variables, index=variables)
    for c in df.columns:
        for r in df.index:
            test_result = grangercausalitytests(data[[r, c]], maxlag= lagmax, verbose=False)
            p_values = [round(test_result[i+1][0][test][1],4) for i in range(lagmax)]
            min_p_value = np.min(p_values)
            df.loc[r, c] = min_p_value
    df.columns = [var + '_x' for var in variables]
    df.index = [var + '_y' for var in variables]
    return df


def fit_model(df, x_vars_list, y_var_list, constant = True, plot_res =False, summary = True, robust_std_err = False):
    X = df[x_vars_list]
    y = df[y_var_list]
    if constant == True:
        X = sm.add_constant(X)
    model = sm.OLS(y,X)
    fit = model.fit()
    if robust_std_err == True:
        fit=  model.fit(cov_type='HAC',cov_kwds={'maxlags':1})
    if summary == True:
        print(fit.summary())
    res = pd.DataFrame(fit.fittedvalues, columns = ['y_est'])
    res = pd.concat([res, y], axis = 1)
    res['residuals'] = res[y_var_list[0]] - res['y_est']
    if plot_res == True:
        res[[y_var_list[0], 'y_est']].plot(figsize = (16, 5), title ='fitted vs observed')
        plot_acf(res['residuals'], lags = 50)
    
    return {
        'results': res, 
        'fit': fit, 
        'paramns': fit.params,
        'R2': fit.rsquared, 
        'SE': fit.bse
    }



def rolling_reg(df, y_var, x_vars, window, intercept = True, plot =True):
    df_ = check_df_order(df, 'ascending')
    exog_fact = df_[x_vars]
    if intercept == True:
        exog_vars = sm.add_constant(exog_fact)
    
    roll_reg = RollingOLS(df_[y_var], exog_vars, window = window)
    res = roll_reg.fit()
    res_params = res.params
    
    df_['const'] = 1
    y_hat_df = res.params * df_
    y_hat_df['y_hat'] = y_hat_df.sum(axis = 1)
    
    final_df = pd.concat([res_params, y_hat_df['y_hat']], axis = 1)
    final_df[y_var] = df_[y_var]
    final_df['res'] = final_df[y_var[0]] - final_df['y_hat']
    final_df = check_df_order(final_df, 'descending')
    final_df['SSR_total'] = (final_df['res']**2).cumsum()   
    final_df['SSR'] = (final_df['res']**2).rolling(window).sum()
    final_df['r_2'] = res.rsquared
    
    final_df = final_df.dropna() 
    
    if plot == True:
        final_df[[y_var[0], 'y_hat']].plot(figsize = [14, 6], title ='fitted vs obs')
    return final_df.dropna()

def optimal_rolling_window(df, y_var, x_vars, roll_window_list, plot = True):
    res_list = []
    for i in roll_window_list:
        tmp = rolling_reg(df, y_var, x_vars, i,  plot = False)
        res_list.append(tmp['SSR'][0])
    res_df = pd.DataFrame(res_list, index = roll_window_list, columns = ['SSR'])
    res_df.index.name ='roll_per'
    best_ = res_df[res_df.values == res_df.min()[0]].index[0]
    print('Optimal Roll. period equal to ' + str(best_))
    res_df.plot(figsize =(14, 6), title = 'SSR vs Rolling Windows')
    return res_df


In [ ]:
from sklearn.linear_model import LassoCV 
from sklearn.preprocessing import StandardScaler 
from statsmodels.stats.outliers_influence import variance_inflation_factor 
from statsmodels.tsa.stattools import grangercausalitytests 
import statsmodels.api as sm


def select_features_lasso(df, target, vif_thresh=5.0, alpha_grid=None, cv_folds=5, return_full_df=True):
    """
    Esegue feature selection con LassoCV + VIF. Restituisce alpha, coefficienti e tabella riassuntiva.

    Parametri:
    -----------
    df : pd.DataFrame
        DataFrame con target e features.
    target : str
        Nome della variabile target.
    vif_thresh : float
        Soglia di VIF per rimozione collinearità.
    alpha_grid : array-like
        Lista di valori alpha da testare.
    cv_folds : int
        Numero di fold per la cross-validation.
    return_full_df : bool
        Se True ritorna un DataFrame con feature, VIF, coeff, flag selezione.

    Ritorna:
    --------
    selected_features : list
    alpha_selected : float
    lasso_coef_series : pd.Series
    full_table (facoltativo): pd.DataFrame con VIF + coeff + selezione
    """

    y = df[target]
    X = df.drop(columns=[target])

    # Standardizzazione
    scaler = StandardScaler()
    X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns, index=X.index)

    # Funzione VIF
    def compute_vif(X):
        vif_data = pd.DataFrame()
        vif_data["feature"] = X.columns
        vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
        return vif_data.set_index("feature")

    # Rimuovi collinearità
    def reduce_vif(X, threshold):
        while True:
            vif = compute_vif(X)
            max_vif = vif['VIF'].max()
            if max_vif > threshold:
                drop_feature = vif['VIF'].idxmax()
                print(f" Rimuovo per VIF alto: {drop_feature} (VIF={max_vif:.2f})")
                X = X.drop(columns=[drop_feature])
            else:
                break
        return X, compute_vif(X)

    X_vif, final_vif = reduce_vif(X_scaled, vif_thresh)

    # LassoCV
    if alpha_grid is None:
        alpha_grid = np.logspace(-4, 0.1, 100)

    lasso = LassoCV(alphas=alpha_grid, cv=cv_folds, random_state=42)
    lasso.fit(X_vif, y)

    alpha_selected = lasso.alpha_
    coef = pd.Series(lasso.coef_, index=X_vif.columns)
    selected_features = list(coef[coef != 0].index)

    print(f"\n Alpha selezionato: {alpha_selected:.5f}")
    print("\n Coefficienti Lasso (solo ≠ 0):")
    print(coef[coef != 0].sort_values(key=abs, ascending=False))

    if return_full_df:
        full_table = final_vif.copy()
        full_table["Lasso_Coefficient"] = coef
        full_table["Selected"] = full_table["Lasso_Coefficient"] != 0
        full_table = full_table.sort_values(by="Selected", ascending=False)
        return selected_features, alpha_selected, coef, full_table
    else:
        return selected_features, alpha_selected, coef 


def get_previous_friday(date):
    return date - timedelta( days = (date.weekday() -4) % 7)

Inputs download

In [ ]:
from_ = dt.datetime(2001, 5, 21)

to_ = dt.datetime.today()

flds = ["PX_LAST"]

period = 'WEEKLY'

Repo- features creation

In [ ]:
transition_dt = dt.datetime(2019, 10, 31)

sprd_adj = 0.082

roll_MA_weeks = 4

O/N repo adj

In [ ]:
## 3M estr, 3M EONIA,
Tickers_repo_3m = ['EESWEC BGN Curncy', 'EUSWEC CFSW Curncy']
cols_3m = ["ESTR_3M", "EONIA_3M"]

## O/N: Estr O/N, Eonia O/N, German GC O/N

Ticker_Repo_ON = ['ESTRON Index', 'EONIA Index', 'REFRGCDE Index']


cols_on = ['ESTR_ON', "EONIA_ON",  "Germ_GC"]

In [ ]:
repo_on_df = bbg_download_clean_df(Ticker_Repo_ON, flds, from_, to_, period)
repo_on_df.columns = cols_on


## Adjusting OIS 

repo_on_df.loc[repo_on_df.index < transition_dt, 'ESTR_ON'] = (
repo_on_df.loc[repo_on_df.index < transition_dt, 'EONIA_ON'] - sprd_adj
)

## Dropping Eonia col bc is now redundant
repo_on_df= repo_on_df.drop('EONIA_ON', axis = 1)
repo_on_df = repo_on_df.dropna()

repo_on_df.head(10)

In [ ]:
### REPO ADJUSTMENT FOR Q/ENDS


res = []
for i in range(len(repo_on_df.index) -5):
    curr = 100*(repo_on_df['Germ_GC'].iloc[i] - repo_on_df['ESTR_ON'].iloc[i])
    lag = 100*(repo_on_df['Germ_GC'].iloc[i + 5] - repo_on_df['ESTR_ON'].iloc[i + 5])
    diff = curr-lag
    
    if i > 0 and abs(diff) > 10:
        curr = res[-1]
    res.append(curr)

## Add to the dataframe
repo_on_df = repo_on_df.iloc[:-5].copy()
repo_on_df['Germ_GC_vs_OIS'] = res

#smoothing repo O/N with moving average
repo_on_df['Germ_GC_vs_OIS'] = repo_on_df['Germ_GC_vs_OIS'].sort_index().rolling(roll_MA_weeks).mean()

repo_on_df = repo_on_df.dropna()    

## Results check:
repo_on_df['Germ_GC_vs_OIS'].plot()

In [ ]:
Germ_CS_cs_OIS_df = repo_on_df[['Germ_GC_vs_OIS']]

QE Variables:

In [ ]:
QE_tickers = ['ECBCPSPP Index', 'ECBCPEPP Index']

EA_GDP = ['ECOXEAS Index']


QE_df = bbg_download_clean_df(QE_tickers, flds, from_, to_, period)
QE_df.columns = ['PSPP_Stock', 'PEPP_Stock']

QE_df['PEPP_Stock'] = QE_df['PEPP_Stock'].fillna(0)

QE_df['Tot_QE_Stock_bn'] = (QE_df['PSPP_Stock'] + QE_df['PEPP_Stock'])/1000

EA GDP

In [ ]:
EA_GDP_df = bbg_download_clean_df(EA_GDP, flds, from_, to_)
EA_GDP_df.columns = ['EA_GDP_bn']

In [ ]:
### Creating final df
QE_final_df = pd.concat([QE_df, EA_GDP_df], axis = 1)
QE_final_df['EA_GDP_bn'] = QE_final_df['EA_GDP_bn'].fillna(method ='ffill')

### Slicing to todays value:
QE_final_df = QE_final_df[QE_final_df.index <= QE_df.index[0]]

#### QE Stock as % of GDP

QE_final_df['QE_Stock_perc_GDP'] = QE_final_df['Tot_QE_Stock_bn']/QE_final_df['EA_GDP_bn']
QE_final_df['QE_Stock_perc_GDP'] = QE_final_df['QE_Stock_perc_GDP'].fillna(method = 'ffill').dropna()
QE_final_df['QE_Stock_perc_GDP'].plot()


In [ ]:
QE_stock_perc_GDP = QE_final_df[['QE_Stock_perc_GDP']]

CISS

In [ ]:
CISS_ticker = ['EASSCOMN Index']
col_CISS = ['CISS']


CISS_df = bbg_download_clean_df(CISS_ticker, flds, from_, to_, period)
CISS_df.columns = col_CISS

Concat Features:

In [ ]:
asw_regressors_df = pd.concat([Germ_CS_cs_OIS_df,  QE_stock_perc_GDP, CISS_df], axis = 1)
asw_regressors_df = asw_regressors_df.fillna(method = 'ffill')

asw_regressors_df = asw_regressors_df.fillna(0)

In [ ]:
asw_regressors_df = asw_regressors_df[asw_regressors_df.index >= dt.datetime(2004, 1, 1)]

Budget Balance 1y out cons

In [ ]:
def cals_weeks_to_year_end(date):
    date = pd.to_datetime(date).normalize()
    end_of_year = pd.Timestamp(date.year, 12, 31)
    delta_days = (end_of_year - date).days
    return int(np.ceil(delta_days / 7))


def get_value_at_date(date, col):
    try:
        return bud_bal_df.loc[date, col] if date in bud_bal_df.index else None
    except:
        return None

In [ ]:
### Downloading Data

final_yr = 27

yr_list = [str(i +1).zfill(2) for i in range(8, final_yr)]

start_date_download = dt.datetime(2009, 1,1)
end_yrs = range(2009, 2000+ final_yr+1)

date_ranges = [(start_date_download, dt.datetime(year, 12, 31)) for year in end_yrs]

ticker_name = "ECBBDE"

bud_balance_tickers = [ticker_name +" " + i +" Index" for i in yr_list]


bud_bal_df = pd.DataFrame()

for i in range(len(bud_balance_tickers)):
    tmp = bbg_download_clean_df(
        bud_balance_tickers[i],
        "PX_LAST",
        date_ranges[i][0], 
        date_ranges[i][-1],
        period
    )
    
    
    tmp.columns = [bud_balance_tickers[i]]
    bud_bal_df = pd.concat([bud_bal_df, tmp], axis = 1)


In [ ]:
start_dates = pd.date_range(
    start = date_ranges[-1][0].strftime('%Y-%m-%d'),
    end = date_ranges[-1][-1].strftime('%Y-%m-%d'),
    freq ='W'
).normalize()

end_dates = start_dates + pd.DateOffset(years = 1)


df = pd.DataFrame({
    "start_date": start_dates,
    "end_date": end_dates
})

df['start_date'] , df['end_date'] = df['start_date'].apply(get_previous_friday), df['end_date'].apply(get_previous_friday)


weeks = []

for i in range(len(df.index)):
    weeks.append(cals_weeks_to_year_end(df['start_date'][i]))
    
df['weeks_to_yr_end'] = weeks

df["weeks to next yr"] = 52 - df["weeks_to_yr_end"]
df["sum_aux"] =  df["weeks to next yr"] + df["weeks_to_yr_end"]
df['w1'] = df["weeks_to_yr_end"]/df["sum_aux"]
df['w2'] = 1 - df['w1']

df['start_date'] , df['end_date'] = df['start_date'].apply(get_previous_friday), df['end_date'].apply(get_previous_friday)

df['col_start'] = df["start_date"].dt.year.astype(str).str[-2:]
df['col_start'] = ticker_name +" "+ df["col_start"] + " Index"

df['col_end'] = df["end_date"].dt.year.astype(str).str[-2:]
df['col_end'] = ticker_name +" " + df["col_end"] + " Index"



df['val_start'] = df.apply(lambda row: get_value_at_date(row["start_date"], row["col_start"]), axis = 1)
df["val_end"] = df.apply(lambda row: get_value_at_date(row["start_date"], row["col_end"]), axis = 1)

df = df.dropna()
df['1yr_bud_bal_avg'] = df['w1']*df['val_start'] + df['w2']*df['val_end']
df = df.set_index ('start_date')

In [ ]:
df_germ_bud_bal = df["1yr_bud_bal_avg"] 
pd.DataFrame(df_germ_bud_bal).resample('M').mean()

Concat everything

In [ ]:
asw_features_df = pd.concat([asw_regressors_df, df_germ_bud_bal], axis = 1).dropna()

Download ASW -> Y vars

In [ ]:
swap_bonds_tickers = [
    'EESWE2 BGN Curncy', 'EESWE5 BGN Curncy', 'EESWE10 BGN Curncy', 'EESWE30 BGN Curncy', 
    'RV0002P 2Y BLC Curncy', 'RV0002P 5Y BLC Curncy', 'RV0002P 10Y BLC Curncy', 'RV0002P 30Y BLC Curncy'

]

tenors = [2, 5, 10, 30]

suffix_swaps = 'y_OIS'
suffix_cash = 'y_Ger'



swap_name = [str(t) + suffix_swaps for t in tenors]
bond_name = [str(t) + suffix_cash for t in tenors]

cols_name_asw = swap_name + bond_name

In [ ]:
asw_df = bbg_download_clean_df(swap_bonds_tickers, flds, from_, to_, period)
asw_df.columns = cols_name_asw
asw_df = asw_df.dropna()

In [ ]:
tuple_list = [(swap_name[i], bond_name[i]) for i in range(len(swap_name))]

Aux Function:

In [ ]:
### Creating ASW vs ESTR:


df_asw = pd.DataFrame()
cols_asw = [str(t) + 'y_ASW' for t in tenors]

for i in range(len(tuple_list)):
    tmp = asw_spread(
        asw_df, tenor_cash = tuple_list[i][-1], tenor_swap = tuple_list[i][0], name = cols_asw[i]
    )
    df_asw = pd.concat([df_asw, tmp], axis = 1)

Aggregating Databases:

In [2]:
database_for_regression = pd.concat([df_asw, asw_features_df], axis = 1)
database_for_regression = database_for_regression.dropna()

NameError: name 'pd' is not defined

In [ ]:
mask = database_for_regression.index > dt.datetime(2013, 1, 1)

database_for_regression = database_for_regression[mask]
database_for_regression


Regression:

In [ ]:
vars_to_grab_dic= {}
features = list(asw_features_df.columns)


for i in cols_asw:
    vars_to_grab_dic[i] = [i] + features

In [ ]:
cols_schatz = vars_to_grab_dic['2y_ASW']

schatz_df = database_for_regression[cols_schatz]

Testing Features:

In [ ]:
features_DU, alpha_DU, coef_DU, full_table_DU = select_features_lasso(schatz_df, '2y_ASW', vif_thresh=5.0, alpha_grid=None, cv_folds=5, return_full_df=True)


In [ ]:
full_table_DU.columns = pd.MultiIndex.from_product([['DU'], full_table_DU.columns])
full_table_DU

In [ ]:
granger_DU = grangers_causation_matrix(schatz_df, list(schatz_df.columns))
granger_DU  = granger_DU[["2y_ASW_x"]]
granger_DU 

Model fitting

In [ ]:
schatz_model = fit_model(
    schatz_df, features, ['2y_ASW'], plot_res = True, robust_std_err= True
)

Saving Res DU:

In [ ]:
DU_coeff = pd.DataFrame(schatz_model['paramns'], columns = ['DU'])
DU_coeff.loc['R2'] = [schatz_model['R2']]

In [ ]:
schatz_model = schatz_model['results'][::-1]
schatz_model.columns = ['2y_ASW_est', '2y_ASW', '2y_asw_residuals']

Bobl:

In [ ]:
cols_bobl = vars_to_grab_dic['5y_ASW']

bobl_df = database_for_regression[cols_bobl]

Testing Features:

In [ ]:
features_OE, alpha_OE, coef_OE, full_table_OE = select_features_lasso(bobl_df, '5y_ASW', vif_thresh=5.0, alpha_grid=None, cv_folds=5, return_full_df=True)


In [ ]:
full_table_OE.columns = pd.MultiIndex.from_product([['OE'], full_table_OE.columns])
full_table_OE

Granger:

In [ ]:
granger_OE = grangers_causation_matrix(bobl_df, list(bobl_df.columns))
granger_OE = granger_OE[["5y_ASW_x"]]
granger_OE

Fitting Model

In [ ]:
bobl_model = fit_model(
    bobl_df, features, ['5y_ASW'], plot_res = True, robust_std_err= True
)

Saving Results

In [ ]:
OE_coeff = pd.DataFrame(bobl_model['paramns'], columns = ['OE'])
OE_coeff.loc['R2'] = [bobl_model['R2']]

In [ ]:
bobl_model = bobl_model['results'][::-1]
bobl_model.columns = ['5y_ASW_est', '5y_ASW', '5y_asw_residuals']

Bund

In [ ]:
cols_bund = vars_to_grab_dic['10y_ASW']

bund_df = database_for_regression[cols_bund]

Testing Features:

In [ ]:
features_RX, alpha_RX, cRXf_RX, full_table_RX = select_features_lasso(bund_df, '10y_ASW', vif_thresh=5.0, alpha_grid=None, cv_folds=5, return_full_df=True)


In [ ]:
features_RX, alpha_RX, cRXf_RX, full_table_RX = select_features_lasso(bund_df, '10y_ASW', vif_thresh=5.0, alpha_grid=None, cv_folds=5, return_full_df=True)


Granger:

In [ ]:
granger_RX = grangers_causation_matrix(bund_df, list(bund_df.columns))
granger_RX  = granger_RX[["10y_ASW_x"]]
granger_RX 

Fitting Model:

In [ ]:
bund_model = fit_model(
    bund_df, features, ['10y_ASW'], plot_res = True, robust_std_err= True
)

Saving results:

In [ ]:
RX_coeff = pd.DataFrame(bund_model['paramns'], columns = ['RX'])
RX_coeff.loc['R2'] = [bund_model['R2']]

In [ ]:
bund_model = bund_model['results'][::-1]
bund_model.columns = ['10y_ASW_est', '10y_ASW', '10y_asw_residuals']

Buxl

In [ ]:
cols_buxl = vars_to_grab_dic['30y_ASW']

buxl_df = database_for_regression[cols_buxl]

Testing features

In [ ]:
cols_buxl = vars_to_grab_dic['30y_ASW']

buxl_df = database_for_regression[cols_buxl]

In [ ]:
full_table_UB.columns = pd.MultiIndex.from_product([['UB'], full_table_UB.columns])
full_table_UB

Granger:

In [ ]:
granger_UB = grangers_causation_matrix(buxl_df, list(buxl_df.columns))
granger_UB = granger_UB[["30y_ASW_x"]]
granger_UB

Model fitting:

In [ ]:
buxl_model = fit_model(
    buxl_df, features, ['30y_ASW'], plot_res = True, robust_std_err= True
)

Saving Results:

In [ ]:
UB_coeff = pd.DataFrame(buxl_model['paramns'], columns = ['UB'])
UB_coeff.loc['R2'] = [buxl_model['R2']]

In [ ]:
buxl_model = buxl_model['results'][::-1]
buxl_model.columns = ['30y_ASW_est', '30y_ASW', '30y_asw_residuals']

Aggregating Results:

In [ ]:
results = pd.concat([schatz_model, bobl_model, bund_model, buxl_model], axis = 1)


In [ ]:
ASW_coeff = pd.concat([DU_coeff, OE_coeff, RX_coeff, UB_coeff], axis = 1)
ASW_coeff.to_clipboard()

In [ ]:
granger_causality = pd.concat([granger_DU, granger_OE, granger_RX,granger_UB], axis = 1)
granger_causality.dropna()

In [ ]:
Lasso_res = pd.concat([full_table_DU, full_table_OE, full_table_RX, full_table_UB], axis = 1)
Lasso_res.to_clipboard()

exporting 

In [ ]:
export_path = "L:\Python Results Export\\Ger_ASW_res.xlsx"

In [ ]:
with pd.ExcelWriter(export_path) as writer:
    ASW_coeff.to_excel(writer, sheet_name = 'Models_Params')
    results.to_excel(writer, sheet_name = 'Est_vs_Fit')
    inputs.to_excel(writer , sheet_name = 'Model_Inputs')
    repo_vs_supply.to_excel(writer, sheet_name = 'Issuance&Repo')